In [ ]:
import pathlib

from functools import lru_cache

import pandas as pd
import swifter

from rdkit import Chem

In [ ]:
cwd = pathlib.Path.cwd()
QUANTUM_GREEN_DIR = pathlib.Path("/home/shared/projects/quantum_green")
PAPER_DIR = QUANTUM_GREEN_DIR / "paper" / "figure" / "section_3_2_3_rate"
DATABASE_DIR = QUANTUM_GREEN_DIR / "datasets_for_publication" / "data" / "kinetics"

In [ ]:
pd.set_option("display.max_columns", None)


def head(df, n=2):
    display(df.head(n))
    print(f"Contains {len(df)} rows")


@lru_cache(maxsize=None)
def canonical_smiles(smiles):
    mol = Chem.MolFromSmiles(smiles)
    for atom in mol.GetAtoms():
        atom.SetAtomMapNum(0)
    return Chem.MolToSmiles(mol, isomericSmiles=True)


def clean_rxn_smi(rxn_smi):
    return ">>".join(
        [
            ".".join([canonical_smiles(smi) for smi in category.split(".")])
            for category in rxn_smi.split(">>")
        ]
    )


def get_rxn_smi(row):
    return row["r1smi"] + "." + row["r2smi"] + ">>" + row["p1smi"] + "." + row["p2smi"]

In [ ]:
rate_data = pd.read_csv(
    PAPER_DIR / "quantum_green_ts_data_24september17_dft_opted_dlpno_sp_rates.csv"
)
rate_data["rxn_smi"] = rate_data.swifter.apply(get_rxn_smi, axis=1)
head(rate_data)

In [ ]:
rate_data["clean_rxn_smi"] = rate_data["rxn_smi"].swifter.apply(clean_rxn_smi)

In [ ]:
zpe_data = pd.read_pickle(PAPER_DIR / "ts_key_characteristics_july31a.pkl")
head(zpe_data)

In [ ]:
kinetics_df = pd.DataFrame(
    {
        "rxn_smi": rate_data["clean_rxn_smi"],
        "k_298": rate_data["k_298"],
        "A_low": rate_data["low_A"],
        "Ea_low": rate_data["low_Ea"],
        "A_high": rate_data["high_A"],
        "Ea_high": rate_data["high_Ea"],
        "barrier": rate_data["rxn_smi"].map(
            zpe_data.set_index("rxn_smi")["fwd_barrier_dlpno_sp_dft_zpe_scaled_kcal"]
        ),
        "Hrxn": rate_data["rxn_smi"].map(
            zpe_data.set_index("rxn_smi")["fwd_Hrxn_dlpno_sp_dft_zpe_scaled_kcal"]
        ),
    }
)
head(kinetics_df)

Looking for duplicates

In [ ]:
kinetics_df[kinetics_df["rxn_smi"].duplicated(keep=False)].sort_values(by="rxn_smi")

In [ ]:
kinetics_df_no_duplicates = kinetics_df.drop_duplicates(subset="rxn_smi", keep=False)

In [ ]:
kinetics_df_no_duplicates.to_csv(
    DATABASE_DIR / "quantumpioneer_kinetics_dataset.csv", index=False
)